# Interactive Demo: Submodular Multi-Document Summarization
### Hui Lin and Jeff Bilmes (2011), ACL-HLT

This notebook demonstrates step-by-step monotone submodular document summarization under length budgets on a real DUC 2004 cluster.

In [ ]:
import sys
from pathlib import Path

# Add src to Python path
if "." not in sys.path:
    sys.path.append(".")

from src.config import ExperimentConfig
from src.data.duc_loader import DUCDatasetLoader
from src.preprocessing.sentence_splitter import SentenceSplitter
from src.preprocessing.normalizer import TextNormalizer
from src.preprocessing.vectorizer import ClusterVectorizer
from src.features.similarity import SimilarityMatrixBuilder
from src.features.centrality import CentralityCalculator
from src.features.clustering import SentenceClusterer
from src.objectives.coverage import TruncatedCoverageObjective
from src.objectives.diversity import ClusterDiversityObjective
from src.objectives.composite import CompositeSubmodularObjective
from src.optimizers.greedy_knapsack import BudgetedGreedyOptimizer
from src.optimizers.baselines import LeadBaselineSummarizer, RandomBaselineSummarizer
from src.evaluation.rouge_eval import RougeEvaluator
from src.evaluation.reports import EvaluationReporter

## 1. Load a DUC Document Cluster

In [ ]:
config = ExperimentConfig.from_yaml("configs/generic_duc03_duc04.yaml")
splitter = SentenceSplitter(min_chars=15, min_words=3)
normalizer = TextNormalizer(lowercase=True, remove_stopwords=True, use_stemming=True)
evaluator = RougeEvaluator(metrics=["rouge1", "rouge2", "rougeL"])

loader = DUCDatasetLoader(
    base_dir="data/duc2004",
    segmenter_fn=splitter.split,
    normalizer_fn=normalizer.normalize,
)

clusters = loader.load_dataset()
cluster = clusters[2]  # Cluster 1001 (South Africa Truth and Reconciliation Commission)
print(f"Loaded Cluster: {cluster.cluster_id}")
print(f"Number of documents: {len(cluster.documents)}")
print(f"Number of sentences: {len(cluster.sentences)}")
print(f"Number of human references: {len(cluster.references)}")

## 2. Feature Extraction & Pairwise Similarity Matrix W

In [ ]:
raw_texts = [s.original_text for s in cluster.sentences]
vectorizer = ClusterVectorizer(normalizer=normalizer, ngram_range=(1, 2), sublinear_tf=True)
tfidf_matrix = vectorizer.fit_transform(raw_texts)

W = SimilarityMatrixBuilder.compute(tfidf_matrix, keep_diagonal=False, clip_negative=True)
r = CentralityCalculator.compute(W)

clusterer = SentenceClusterer(k_ratio=0.10, random_seed=42)
labels, partitions = clusterer.cluster(tfidf_matrix)

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"Similarity Matrix W Shape: {W.shape} (Symmetric: {SimilarityMatrixBuilder.validate_properties(W)})")
print(f"Number of Clusters K: {len(partitions)}")

## 3. Submodular Maximization: F(S) = L_1(S) + lambda * R_1(S)

In [ ]:
alpha = 0.20
lambda_val = 4.0
budget_words = 100

cov_obj = TruncatedCoverageObjective(W=W, alpha=alpha)
div_obj = ClusterDiversityObjective(cluster_labels=labels, num_clusters=len(partitions), rewards=r)
comp_obj = CompositeSubmodularObjective(coverage_obj=cov_obj, diversity_objs=[(lambda_val, div_obj)])

optimizer = BudgetedGreedyOptimizer(budget_limit=budget_words, budget_type="words", sort_order="document_order")
result = optimizer.optimize(comp_obj, cluster.sentences)

print(f"Selected {result.num_sentences} sentences ({result.total_cost}/{budget_words} words):")
print("-" * 80)
print(result.summary_text)
print("-" * 80)
print("Objective breakdown:", result.breakdown)

## 4. Evaluation and Baseline Comparison

In [ ]:
lead_opt = LeadBaselineSummarizer(budget_limit=budget_words, budget_type="words")
lead_res = lead_opt.summarize(cluster.sentences)

rand_opt = RandomBaselineSummarizer(budget_limit=budget_words, budget_type="words", seed=42)
rand_res = rand_opt.summarize(cluster.sentences)

scores_proposed = evaluator.evaluate(result.summary_text, cluster.references)
scores_lead = evaluator.evaluate(lead_res.summary_text, cluster.references)
scores_rand = evaluator.evaluate(rand_res.summary_text, cluster.references)

import pandas as pd
df_comparison = pd.DataFrame([
    {"Method": "Proposed Submodular (Lin & Bilmes)", "ROUGE-1 R": scores_proposed["rouge1_recall"], "ROUGE-1 F1": scores_proposed["rouge1_f1"], "ROUGE-2 R": scores_proposed["rouge2_recall"], "Redundancy": EvaluationReporter.compute_redundancy(result.selected_indices, W)},
    {"Method": "Lead Baseline", "ROUGE-1 R": scores_lead["rouge1_recall"], "ROUGE-1 F1": scores_lead["rouge1_f1"], "ROUGE-2 R": scores_lead["rouge2_recall"], "Redundancy": EvaluationReporter.compute_redundancy(lead_res.selected_indices, W)},
    {"Method": "Random Baseline", "ROUGE-1 R": scores_rand["rouge1_recall"], "ROUGE-1 F1": scores_rand["rouge1_f1"], "ROUGE-2 R": scores_rand["rouge2_recall"], "Redundancy": EvaluationReporter.compute_redundancy(rand_res.selected_indices, W)},
])
df_comparison